In [3]:
import nltk
import re
import math
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# --- SETUP ---
def initialize_nltk():
    resources = ['punkt', 'stopwords', 'averaged_perceptron_tagger']
    for res in resources:
        nltk.download(res, quiet=True)
    return set(stopwords.words('english')), PorterStemmer()

def write_stage(filename, label, data):
    with open(filename, "a", encoding="utf-8") as f:
        f.write(f"{label}: {data}\n")

def clean_text(text, stop_words, stemmer, label):
    # Tokenization
    tokens = word_tokenize(text)
    write_stage("tokenization.txt", label, tokens)

    # Remove punctuation + lowercase
    tokens_clean = [re.sub(r"[^\w\s]", "", t).lower().strip() for t in tokens]
    write_stage("punctuation_lowercase.txt", label, tokens_clean)

    # Stop word removal + remove numbers/empties
    tokens_no_stop = [
        t for t in tokens_clean
        if t and not re.search(r'\d', t) and t not in stop_words
    ]
    write_stage("stopword_removal.txt", label, tokens_no_stop)

    # Stemming
    stemmed = [stemmer.stem(t) for t in tokens_no_stop]
    write_stage("stemming.txt", label, stemmed)

    # POS Tagging
    pos_tags = nltk.pos_tag(tokens_no_stop)
    write_stage("pos_tags.txt", label, pos_tags)

    return stemmed


# --- MATH ---
def calculate_tfidf(all_docs):
    vocab = sorted(set([t for doc in all_docs for t in doc]))
    N = len(all_docs)

    tfs = []
    for doc in all_docs:
        tfs.append({w: doc.count(w) for w in vocab if w in doc})

    idf = {
        w: math.log((N + 1) / (sum(1 for doc in all_docs if w in doc) + 1)) + 1
        for w in vocab
    }

    vectors = []
    for counts in tfs:
        vectors.append({w: counts.get(w, 0) * idf[w] for w in vocab})
    return vectors


def get_similarity(vec_a, vec_b):
    dot = sum(vec_a.get(k, 0) * vec_b.get(k, 0) for k in vec_a if k in vec_b)
    norm_a = math.sqrt(sum(v**2 for v in vec_a.values()))
    norm_b = math.sqrt(sum(v**2 for v in vec_b.values()))
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0


# --- MAIN ---
if __name__ == "__main__":
    stop_words, stemmer = initialize_nltk()

    # Clear old stage files at start
    stage_files = [
        "tokenization.txt",
        "punctuation_lowercase.txt",
        "stopword_removal.txt",
        "stemming.txt",
        "pos_tags.txt"
    ]
    for f in stage_files:
        open(f, "w").close()

    # Load requirements
    with open('requirements-3nfr-60fr.txt', 'r', encoding='utf-8') as file:
        lines = [line.strip() for line in file if line.strip()]

    nfr_raw = lines[:3]
    fr_raw = lines[3:]

    # Clean with labels
    nfr_docs = [
        clean_text(text, stop_words, stemmer, f"NFR{i+1}")
        for i, text in enumerate(nfr_raw)
    ]

    fr_docs = [
        clean_text(text, stop_words, stemmer, f"FR{i+1}")
        for i, text in enumerate(fr_raw)
    ]

    # Vectorize
    all_vecs = calculate_tfidf(nfr_docs + fr_docs)
    nfr_vecs = all_vecs[:3]
    fr_vecs = all_vecs[3:]

    THRESHOLD = 0.17

    with open("test_output.txt", "w") as f:
        for i, fr_v in enumerate(fr_vecs):
            sims = [get_similarity(fr_v, n_v) for n_v in nfr_vecs]
            binary_row = [1 if s >= THRESHOLD else 0 for s in sims]
            line = f"FR{i+1},{binary_row[0]},{binary_row[1]},{binary_row[2]}"
            f.write(line + "\n")
